# 02_data_cleaning.ipynb
Clean the synthetic datasets by checking missing values, removing duplicates, fixing data types, and handling outliers. This notebook runs independently.

In [4]:
from pathlib import Path
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from faker import Faker
warnings.filterwarnings('ignore')
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DATA_DIR = BASE_DIR / 'data' / 'processed'
GENERATED_DATA_DIR = BASE_DIR / 'data' / 'generated'
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import src.generate_data as generate_data
import src.cleaning as cleaning
print('Setup complete')

Setup complete


## Setup and Import Dependencies
Import necessary libraries and configure directory paths for accessing generated data and saving cleaned datasets. This ensures the notebook has everything needed to run independently.

In [ ]:
# Ensure generated data exists before cleaning
csv_files = sorted(GENERATED_DATA_DIR.glob('*.csv'))
if not csv_files:
    generate_data.generate_all(n_each=1000, out_dir=GENERATED_DATA_DIR)
    csv_files = sorted(GENERATED_DATA_DIR.glob('*.csv'))
[p.name for p in csv_files]

['barangay_census.csv',
 'medical_records.csv',
 'sales_orders.csv',
 'school_enrollment.csv']

## Load Raw Datasets
Locate and load the four generated CSV files from the previous notebook. If they don't exist, this cell regenerates them automatically. This ensures data is always available.

In [ ]:
# Load all datasets
datasets = {p.stem: pd.read_csv(p) for p in csv_files}
{name: df.shape for name, df in datasets.items()}

{'barangay_census': (1020, 7),
 'medical_records': (1020, 10),
 'sales_orders': (1020, 10),
 'school_enrollment': (1020, 10)}

## Load into Dictionary for Processing
Convert all CSV files into a dictionary of pandas DataFrames with dataset names as keys. This makes it easy to process all datasets using the same cleaning functions.

In [ ]:
# Check missing values
for name, df in datasets.items():
    print(f'\n=== Missing values: {name} ===')
    display(cleaning.report_missing(df).rename('missing_count').to_frame())


=== Missing values: barangay_census ===


,missing_count
households,50
population,50
median_income,50
record_id,0
barangay,0
city,0
collection_date,0



=== Missing values: medical_records ===


,missing_count
age,50
temperature_c,50
cholesterol_mgdl,50
weight_kg,50
visit_id,0
visit_date,0
patient_name,0
sex,0
barangay,0
diagnosis,0



=== Missing values: sales_orders ===


,missing_count
quantity,50
unit_price,50
total_amount,50
order_id,0
order_date,0
customer_name,0
barangay,0
product,0
category,0
payment_method,0



=== Missing values: school_enrollment ===


,missing_count
math_score,50
english_score,50
science_score,50
attendance_rate,50
student_id,0
student_name,0
birth_date,0
grade_level,0
school_name,0
barangay,0


## Detect Missing Values  
Identify and report missing values in each dataset. This shows which columns need attention during the cleaning process.

In [ ]:
# Clean sales orders as the main example
sales = datasets['sales_orders'].copy()
sales['order_date'] = pd.to_datetime(sales['order_date'], errors='coerce')
sales['quantity'] = pd.to_numeric(sales['quantity'], errors='coerce')
sales['unit_price'] = pd.to_numeric(sales['unit_price'], errors='coerce')
sales['total_amount'] = pd.to_numeric(sales['total_amount'], errors='coerce')
sales, removed_dupes = cleaning.drop_duplicates(sales)
sales = cleaning.fill_numeric_with_median(sales, ['quantity', 'unit_price', 'total_amount'])
for column in ['quantity', 'unit_price', 'total_amount']:
    sales = cleaning.cap_outliers_iqr(sales, column)
print('Removed duplicates:', removed_dupes)
display(sales.head())

Removed duplicates: 20


,order_id,order_date,customer_name,barangay,product,category,quantity,unit_price,total_amount,payment_method
0,SO-100383,2026-05-24,Elijah Young,Maligaya,Soap,Beverage,3.0,281.92,845.76,Cash
1,SO-100775,2026-05-24,Danny Arnold,San Miguel,Instant Noodles,Grocery,3.0,447.94,1343.82,GCash
2,SO-100791,2026-05-24,Robert Richardson,Rizal,Instant Noodles,Beverage,5.0,523.34,2616.70,Card
3,SO-100131,2026-05-24,Jason Gonzalez,Bagong Silang,Canned Sardines,Household,5.0,573.32,2866.60,Bank Transfer
4,SO-100464,2026-05-24,Colleen Stewart,San Jose,Rice,Household,2.0,738.04,1476.08,Card


## Clean Sales Orders Dataset
Apply comprehensive cleaning to the sales data: convert date/numeric fields to proper types, remove duplicates, fill missing values using median imputation, and cap outliers using the IQR method. This transforms raw data into a reliable dataset.

In [ ]:
# Save cleaned datasets
for name, df in datasets.items():
    cleaned = df.copy()
    if name == 'sales_orders':
        cleaned = sales
    cleaned.to_csv(PROCESSED_DATA_DIR / f'{name}_cleaned.csv', index=False)
print('Saved cleaned files to data/processed')

Saved cleaned files to data/processed


## Save Cleaned Datasets
Export all cleaned datasets to CSV files in the `data/processed/` directory. These cleaned files are ready for exploratory analysis and visualization.

## Cleaning notes
The notebook now produces cleaned CSV files in `data/processed/` and can be rerun independently.